# SQLMate: train and evaluate on a free Colab GPU

Run the cells from top to bottom. Before you start: **Runtime → Change runtime type → T4 GPU → Save**.

What this notebook does:
1. Measures the base model (Qwen2.5-Coder-1.5B-Instruct) on both test sets.
2. Fine-tunes it with QLoRA (about 45–75 minutes on a T4).
3. Measures the fine-tuned model the same way.
4. Merges the adapter, converts the model to GGUF for Ollama, and (optionally) uploads it to Hugging Face.
5. Packs the results into a zip for you to download and commit.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and install the libraries (2–4 minutes)

In [ ]:
REPO = "https://github.com/TheCraid/sqlmate"   # change if your repo has another name
!git clone -q {REPO} sqlmate || true
%cd /content/sqlmate
!git pull -q
!pip install -q -r requirements-train.txt
!ls -la data

## 1b. Keep progress on Google Drive

Training saves a checkpoint every 25 steps and the results after each test into your Google Drive
(folder `sqlmate-outputs`). If Colab disconnects, run cells 1, 1b, 2, 3 again and then the training cell:
it continues from the last checkpoint. Click **Connect to Google Drive** and allow access when asked.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/sqlmate-outputs"
!mkdir -p {SAVE}/results
!rm -rf outputs && ln -s {SAVE} outputs
!cp -n {SAVE}/results/*.json results/ 2>/dev/null; ls results

## 2. Tokens (optional)

In the left sidebar click the **key icon (Secrets)** and add:
- `HF_TOKEN`: a Hugging Face token with **write** access (needed only for uploading the model in step 7).
- `WANDB_API_KEY`: only if you want training curves on Weights & Biases.

Turn on "Notebook access" for each one. Skip this step if you don't want either.

In [ ]:
import os
REPORT_TO = "none"
try:
    from google.colab import userdata
    for name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[name] = userdata.get(name)
            print(name, "loaded")
        except Exception:
            print(name, "not set (that's fine)")
    if os.environ.get("WANDB_API_KEY"):
        REPORT_TO = "wandb"
except ImportError:
    print("Not running on Colab")
print("Training logs go to:", REPORT_TO)

## 3. Build the training file

In [ ]:
!python -m sqlmate.build_dataset

## 4. Evaluate the base model (5–10 minutes)

In [ ]:
!python -m sqlmate.evaluate --backend hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct --label "Qwen2.5-Coder-1.5B (base)" --kind base
!cp results/*.json {SAVE}/results/


## 5. Fine-tune with QLoRA (about 1.5 hours)

150 steps × 16 examples. A free T4 manages about one step every 35 seconds, so a full pass over all the
data does not fit in one free session; 150 steps does. Keep this tab open. If Colab disconnects, run
cells 1, 1b, 2, 3 and this one again: it continues from the last checkpoint on your Drive.

In [ ]:
!python -m sqlmate.train --max-steps 150 --report-to {REPORT_TO}


## 6. Evaluate the fine-tuned model (5–10 minutes) and make the report

In [ ]:
!python -m sqlmate.evaluate --backend hf --model Qwen/Qwen2.5-Coder-1.5B-Instruct --adapter outputs/sqlmate-lora --label "SQLMate-1.5B (fine-tuned)" --kind finetuned
!python -m sqlmate.report
!cp results/*.json results/*.png {SAVE}/results/


In [ ]:
from IPython.display import Image
Image("results/accuracy.png")

## 7. Merge, convert to GGUF, upload (15–25 minutes)

Set `HF_REPO` to your Hugging Face username and a model name. Leave it empty to skip uploading.

In [ ]:
HF_REPO = "TheCraid/sqlmate-1.5b"   # set to "" to skip uploading
push = f"--push {HF_REPO}" if HF_REPO and os.environ.get("HF_TOKEN") else ""
!python -m sqlmate.merge_export {push}

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -e llama.cpp/gguf-py sentencepiece
!python llama.cpp/convert_hf_to_gguf.py outputs/sqlmate-merged --outfile sqlmate-q8_0.gguf --outtype q8_0
!ls -lh sqlmate-q8_0.gguf

In [ ]:
if HF_REPO and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    api = HfApi()
    gguf_repo = HF_REPO + "-gguf"
    api.create_repo(gguf_repo, exist_ok=True)
    api.upload_file(path_or_fileobj="sqlmate-q8_0.gguf", path_in_repo="sqlmate-q8_0.gguf", repo_id=gguf_repo)
    api.upload_file(path_or_fileobj="ollama/Modelfile", path_in_repo="Modelfile", repo_id=gguf_repo)
    print(f"GGUF uploaded: https://huggingface.co/{gguf_repo}")
else:
    print("Skipped upload. Download sqlmate-q8_0.gguf from the Files panel instead (about 1.6 GB).")

## 8. Download the results

This downloads `sqlmate_results.zip` (results and the small LoRA adapter). Unzip it into your laptop's `sqlmate` folder.

In [ ]:
!zip -qr sqlmate_results.zip results outputs/sqlmate-lora/adapter_config.json outputs/sqlmate-lora/adapter_model.safetensors outputs/sqlmate-lora/training_summary.json
from google.colab import files
files.download("sqlmate_results.zip")